# Transformer Token Attribution for TLDC

This notebook demonstrates token occlusion and Integrated Gradients for a locally available transformer checkpoint. Attention weights alone are not treated as faithful explanations. Token masking and padding baselines can create linguistically or legally invalid text, so attribution requires expert review.

The notebook is opt-in and loads only local checkpoint files from `models/xlm_roberta`; it does not download model weights. Install optional packages using `requirements-xai.txt`.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CHECKPOINT_PATH = ROOT / "models/xlm_roberta"
TEST_PATH = ROOT / "data/test/tldc_test.jsonl"
SEGMENTS_PATH = ROOT / "annotations/document_segmentation/segmented_documents.jsonl"
OUTPUT_PATH = ROOT / "xai/integrated_gradients/transformer_attributions.jsonl"
RUN_TRANSFORMER_ATTRIBUTION = False


def read_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]


test_rows = [row for row in read_jsonl(TEST_PATH) if row.get("task") == "document_classification"]
if not RUN_TRANSFORMER_ATTRIBUTION:
    print("Transformer attribution is disabled; no checkpoint download will occur.")
elif not CHECKPOINT_PATH.exists() or not test_rows:
    print("Skip transformer attribution: a trained local checkpoint and reviewed held-out example are required.")
else:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from src.explainability.transformer_attribution import (
        integrated_gradients_attributions,
        token_occlusion_attributions,
    )
    from src.explainability.evidence_linker import link_attributions

    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_PATH, local_files_only=True)
    if torch.cuda.is_available():
        model = model.cuda()
    example = test_rows[0]
    source_documents = {row["document_id"]: row for row in read_jsonl(SEGMENTS_PATH)}
    source_document = source_documents.get(example["document_id"], {})
    occlusion = token_occlusion_attributions(model, tokenizer, example["text"])
    label = model.config.id2label.get(occlusion["target_index"], str(occlusion["target_index"]))
    record = link_attributions(
        example["document_id"], example["text"], source_document.get("sections", []),
        occlusion["attributions"], label, str(CHECKPOINT_PATH.name), occlusion["method"],
        score=occlusion["target_score"],
    )
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("a", encoding="utf-8") as output:
        output.write(json.dumps(record, ensure_ascii=False) + "\n")
    record

Transformer attribution is disabled; no checkpoint download will occur.


In [2]:
# Integrated Gradients is an independent attribution method; compare it with occlusion rather than treating either as proof.
if RUN_TRANSFORMER_ATTRIBUTION and "model" in globals() and test_rows:
    integrated = integrated_gradients_attributions(model, tokenizer, example["text"])
    ig_label = model.config.id2label.get(integrated["target_index"], str(integrated["target_index"]))
    ig_record = link_attributions(
        example["document_id"], example["text"], source_document.get("sections", []),
        integrated["attributions"], ig_label, str(CHECKPOINT_PATH.name), integrated["method"],
    )
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("a", encoding="utf-8") as output:
        output.write(json.dumps(ig_record, ensure_ascii=False) + "\n")
    print("Integrated Gradients convergence delta:", integrated["convergence_delta"])
else:
    print("Integrated Gradients awaits an enabled run with a local model and reviewed test example.")

Integrated Gradients awaits an enabled run with a local model and reviewed test example.
